<a href="https://colab.research.google.com/github/Suim00/hdac-knhanes-skipbreakfast-obesity/blob/main/code/12_sensitivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12주차 민감도분석 (9팀)
국민건강영양조사 제9기(2022-2024) 중 2024년 자료. 11주차 주 분석(모형 2: 아침 결식 → 비만, 복합표본 로지스틱, DAG 보정집합)을 그대로 다시 실행한 뒤, **한 번에 가정 하나만** 바꾸는 민감도분석을 한다.

| 절 | 내용 |
|---|---|
| 1~3 | 11주차 노트북의 불러오기, 변수 만들기, `in_study`(= 안내문의 `in_analysis`), 완전 사례 `cc` (그대로) |
| 4 | 설계 기반 회귀 함수 (11주차 그대로 + 포아송 회귀 한 가지 추가) |
| 5 | 주 분석(모형 2) 재실행과 11주차 결과 일치 확인 |
| 6 | **S-유병률** 포아송(로그 연결) 유병률비 (필수) |
| 7 | **S1** 역방향 인과: 체중 조절 중인 사람을 뺀 도메인 |
| 8 | **S2** 노출 정의: 아침식사 빈도 네 범주 |
| 9 | **S3 (선택)** 직접효과: 에너지 섭취 추가 |
| 10 | **D-2** Table 3·각주, **D-3** 해석 초안, 제출 전 점검 |


## 1. 내 Google Drive 연결 및 자료 읽기 (11주차와 같은 코드)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/hdac_data/HN24_ALL(SAS)'
DATA_FILE = f'{DATA_DIR}/hn24_all.sas7bdat'

Mounted at /content/drive


In [2]:
!pip -q install pyreadstat
import pyreadstat, numpy as np, pandas as pd

need = ['age', 'sex', 'HE_prg', 'L_BR_FQ', 'HE_ht', 'HE_wt', 'HE_BMI', 'HE_obe',
        'wt_tot', 'kstrata', 'psu', 'town_t', 'educ', 'sm_presnt', 'dr_month', 'N_DIET', 'N_EN']

_, meta0 = pyreadstat.read_sas7bdat(DATA_FILE, metadataonly=True)
lower_map = {c.lower(): c for c in meta0.column_names}
present = {v: lower_map.get(v.lower()) for v in need}
assert all(present.values()), [v for v, a in present.items() if a is None]

df, meta = pyreadstat.read_sas7bdat(DATA_FILE, usecols=list(present.values()))
df = df.rename(columns={a: v for v, a in present.items()})
print(df.shape)

(6997, 17)


In [3]:
EXTRA = {'incm': 'incm',               # 소득 사분위(개인) 1~4   ← 정의표 변수명 확인
         'pa_aerobic': 'pa_aerobic'}   # 유산소 신체활동 0/1     ← 정의표 변수명 확인

extra_present = {v: lower_map.get(a.lower()) for v, a in EXTRA.items()}
assert all(extra_present.values()), [v for v, a in extra_present.items() if a is None]

ex, _ = pyreadstat.read_sas7bdat(DATA_FILE, usecols=list(extra_present.values()))
assert len(ex) == len(df)                       # 같은 파일이므로 행 순서가 같다
for v, a in extra_present.items():
    df[v] = ex[a].values

# 비교용 가중치 (정의표의 wt_itvex). 없으면 결측으로 둔다.
wt_alt = lower_map.get('wt_itvex')
if wt_alt:
    df['wt_itvex'] = pyreadstat.read_sas7bdat(DATA_FILE, usecols=[wt_alt])[0][wt_alt].values
else:
    df['wt_itvex'] = np.nan
    print('자료에 wt_itvex 열이 없습니다 (비교 생략)')
print(df[list(EXTRA) + ['wt_itvex']].shape)

(6997, 3)


### 12주차 추가 변수 읽기 (체중 조절 여부)
S1(역방향 인과)에 필요한 변수(`BO2_1`)만 같은 자료에서 더 읽는다. 11주차 `need` 목록은 건드리지 않는다.

변수는 `BO2_1`(최근 1년간 체중 조절 여부)이다. 이용지침서(141~142쪽) 기준 코드값은 1 = 체중 감소 노력, 2 = 체중 유지 노력, 3 = 체중 증가 노력, 4 = 체중 조절 노력해본 적 없음, 8 = 비해당, 9 = 모름·무응답이다. (`BO1_1`은 체중 *변화* 여부라 다른 변수이다.) 정의표와 다르게 정하면 아래 `WL_VAR`와 7절의 `WL_LOSE`만 바꾼다.

In [4]:
WL_VAR = 'BO2_1'     # 1년간 체중 조절 여부 (이용지침서 141쪽). BO1_1은 체중 변화 여부이므로 쓰지 않는다

wl_col = lower_map.get(WL_VAR.lower())
if wl_col:
    df['wl_raw'] = pyreadstat.read_sas7bdat(DATA_FILE, usecols=[wl_col])[0][wl_col].values
else:
    df['wl_raw'] = np.nan
    print(f'자료에 {WL_VAR} 열이 없습니다 → 변수명을 확인한 뒤 다시 실행 (S1만 영향)')
print(f'{WL_VAR} 전체 응답 분포(집계, 1 감소·2 유지·3 증가·4 노력 안 함·8 비해당·9 모름):')
print(df['wl_raw'].value_counts(dropna=False).sort_index().to_string())

BO2_1 전체 응답 분포(집계, 1 감소·2 유지·3 증가·4 노력 안 함·8 비해당·9 모름):
wl_raw
1.0    2525
2.0    1530
3.0     406
4.0    2140
8.0     172
9.0     215
NaN       9


## 2. 분석용 변수 만들기 (11주차와 같은 코드)
원본 행은 지우지 않고 표시변수만 만든다. 아래 두 셀은 11주차 코드 그대로이고, 그다음 셀에서 12주차에 새로 필요한 변수만 만든다.

In [5]:
df['breakfast_skip'] = df['L_BR_FQ'].map({1: 0, 2: 0, 3: 1, 4: 1})
df['obese'] = np.where(df['HE_obe'].isna(), np.nan, (df['HE_obe'] >= 4).astype(float))
df['educ_c'] = df['educ'].where(~df['educ'].isin([88, 99]))
df['edu4'] = df['educ_c'].map({1: 1, 2: 1, 3: 1, 4: 2, 5: 3, 6: 4, 7: 4, 8: 4})   # 묶음은 팀이 확정
df['N_DIET_c'] = df['N_DIET'].where(~df['N_DIET'].isin([9]))
df['clu'] = df['kstrata'].astype(str) + '_' + df['psu'].astype(str)   # 집락 식별자 (층 + 조사구)

In [6]:
df['incm_c'] = df['incm'].where(df['incm'].isin([1, 2, 3, 4]))          # 1=하 … 4=상 (사분위)
df['pa_c'] = df['pa_aerobic'].where(df['pa_aerobic'].isin([0, 1]))      # 1=실천, 0=비실천

### 12주차 추가 변수 만들기
- `brfq`: 아침식사 빈도 네 범주(1~4) 그대로 (S2). 그 밖의 값은 결측.
- `wl_c`: 체중 조절 여부 1~4 외(8·9)는 결측 (S1).
- `en100`: 총에너지 섭취 `N_EN`을 100 kcal 단위로 (S3).

In [7]:
df['brfq'] = df['L_BR_FQ'].where(df['L_BR_FQ'].isin([1, 2, 3, 4]))
df['wl_c'] = df['wl_raw'].where(df['wl_raw'].isin([1, 2, 3, 4]))
df['en100'] = df['N_EN'] / 100

### 2-2. 분석 대상 표시변수 `in_study` (11주차와 같은 코드)

In [8]:
conds = [
    ('1. 만 19세 이상', df['age'] >= 19, '연구 대상 아님(비해당)'),
    ('2. 임신 중 아님', df['HE_prg'] != 1, '제외 기준(임신)'),
    ('3. 아침식사 빈도 응답', df['L_BR_FQ'].isin([1, 2, 3, 4]), '결측(노출 정의 불가)'),
    ('4. 신장·체중·BMI 값 있음', df[['HE_ht', 'HE_wt', 'HE_BMI']].notna().all(axis=1),
     '결측(BMI 이상값 기준은 적용하지 않음)'),
    ('5. 가중치·층·집락 유효',
     df['wt_tot'].notna() & (df['wt_tot'] > 0) & df['kstrata'].notna() & df['psu'].notna(),
     '설계상 제외'),
]

cur = pd.Series(True, index=df.index)
rows = [{'단계': '0. 전체 조사 참여자', '제외 인원': 0, '남은 인원': int(cur.sum()), '사유': '출발 대상'}]
for name, cond, why in conds:
    before = int(cur.sum())
    cur = cur & cond
    rows.append({'단계': name, '제외 인원': before - int(cur.sum()), '남은 인원': int(cur.sum()), '사유': why})

df['in_study'] = cur.astype(int)      # 분석 대상 표시변수
df['design_ok'] = conds[-1][1]        # 설계 변수 유효 표시변수
flow = pd.DataFrame(rows)
print(flow.to_string(index=False))

base = (df['age'] >= 19) & (df['HE_prg'] != 1) & df['L_BR_FQ'].isin([1, 2, 3, 4])
print()
print('4단계 변수별 결측:', df.loc[base, ['HE_ht', 'HE_wt', 'HE_BMI']].isna().sum().to_dict())

               단계  제외 인원  남은 인원                      사유
     0. 전체 조사 참여자      0   6997                   출발 대상
      1. 만 19세 이상    964   6033           연구 대상 아님(비해당)
       2. 임신 중 아님     16   6017               제외 기준(임신)
    3. 아침식사 빈도 응답    174   5843            결측(노출 정의 불가)
4. 신장·체중·BMI 값 있음     94   5749 결측(BMI 이상값 기준은 적용하지 않음)
   5. 가중치·층·집락 유효      0   5749                  설계상 제외

4단계 변수별 결측: {'HE_ht': 94, 'HE_wt': 23, 'HE_BMI': 94}


In [9]:
d = df[df['design_ok']].copy()        # 설계 정보가 있는 전체 (도메인 분석용)
print('설계 자료 n:', len(d), '/ 분석 대상(도메인) n:', int(d['in_study'].sum()))
print('층 수:', d['kstrata'].nunique(), '/ 집락(PSU) 수:', d['clu'].nunique())
psu_per_strata = d.groupby('kstrata')['clu'].nunique()
print('층당 PSU 수 최소:', int(psu_per_strata.min()), '(2 이상이어야 분산 추정 가능)')
print('가중치 합(추정 모집단, 명):', round(d['wt_tot'].sum()))

설계 자료 n: 6805 / 분석 대상(도메인) n: 5749
층 수: 27 / 집락(PSU) 수: 192
층당 PSU 수 최소: 2 (2 이상이어야 분산 추정 가능)
가중치 합(추정 모집단, 명): 51522035


## 3. 보정변수 목록과 완전 사례 (11주차와 같은 코드)
`ADJ_SET`은 11주차 DAG에서 정한 총효과 보정집합 8개이다. **민감도분석에서도 바꾸지 않는다.** 완전 사례 `cc`(5,272명이어야 한다)가 주 분석의 분석 인원이다.

In [10]:
# DAG(부록 A)에서 정한 총효과 보정집합 = 교란변수 8개.  키: (표시 이름, 데이터 열, 모형에서의 형태)
ADJ_SET = {
    'age':   ('나이',          'age',       '연속'),     # 1세당
    'sex':   ('성별',          'sex',       '범주'),
    'educ':  ('교육수준',      'edu4',      '범주'),
    'incm':  ('소득',          'incm_c',    '범주'),
    'area':  ('거주지역',      'town_t',    '범주'),
    'smoke': ('현재흡연',      'sm_presnt', '범주'),
    'drink': ('월간음주',      'dr_month',  '범주'),
    'pa':    ('신체활동(유산소)', 'pa_c',    '범주'),     # 역할이 애매한 변수 (모형 3에서 제외)
}
ADJ_KEYS = list(ADJ_SET)
ADJ_CONT = [v[1] for v in ADJ_SET.values() if v[2] == '연속']     # 연속형 (기준 없음)
ADJ_CAT = [v[1] for v in ADJ_SET.values() if v[2] == '범주']      # 범주형 (가장 작은 값이 기준)
ADJ = ADJ_CONT + ADJ_CAT
assert all(c in d.columns for c in ADJ), [c for c in ADJ if c not in d.columns]

In [11]:
WT = 'wt_tot'

d['cc'] = ((d['in_study'] == 1) & d['breakfast_skip'].notna() & d['obese'].notna()
           & d[ADJ].notna().all(axis=1) & (d[WT] > 0)).astype(int)

n_study, n_cc = int(d['in_study'].sum()), int(d['cc'].sum())
print(f'Table 1 분석 대상(in_study): {n_study:,}명')
print(f'완전 사례(cc, 보정변수 모두 있음): {n_cc:,}명 → 차이 {n_study - n_cc:,}명 ({(n_study - n_cc) / n_study * 100:.1f}%)')
print()
s = d[d['in_study'] == 1]
miss = s[['breakfast_skip', 'obese'] + ADJ].isna().sum()
print('분석 대상 안의 변수별 결측 n:')
print(miss[miss > 0].to_string() if (miss > 0).any() else '  결측 없음')
print()
print('새 변수 코드 점검 (분석 대상, 결측 포함 인원 분포):')
for v in ['incm_c', 'pa_c']:
    print(s[v].value_counts(dropna=False).sort_index().to_string(), '\n')
print('완전 사례의 노출별 인원:')
print(d.loc[d['cc'] == 1, 'breakfast_skip'].value_counts().sort_index().rename({0: '아침 식사(0)', 1: '결식(1)'}).to_string())
w_ = d.loc[d['cc'] == 1]
prev = (w_['wt_tot'] * w_['obese']).sum() / w_['wt_tot'].sum() * 100
print(f"\n완전 사례의 가중 비만 유병률(점추정): {prev:.1f}%  (흔한 결과 → 오즈비 해석 주의, 유병률비는 12주차)")

# 가중치 비교 (정의표 wt_itvex와 10주차 wt_tot): 집계값만 출력
if d['wt_itvex'].notna().any():
    both = (d['wt_tot'] > 0) & (d['wt_itvex'] > 0)
    print('\n가중치 비교 (설계 자료 d 기준)')
    print(f"  wt_tot > 0: {int((d['wt_tot'] > 0).sum()):,}명 / wt_itvex > 0: {int((d['wt_itvex'] > 0).sum()):,}명 / 둘 다: {int(both.sum()):,}명")
    print(f"  분석 대상 안에서 wt_itvex가 결측이거나 0인 사람: {int(((d['in_study'] == 1) & ~(d['wt_itvex'] > 0)).sum()):,}명")
    print(f"  가중치 합(명): wt_tot {d['wt_tot'].sum():,.0f} / wt_itvex {d['wt_itvex'].sum():,.0f}")

Table 1 분석 대상(in_study): 5,749명
완전 사례(cc, 보정변수 모두 있음): 5,272명 → 차이 477명 (8.3%)

분석 대상 안의 변수별 결측 n:
edu4         148
incm_c        28
sm_presnt    141
dr_month     140
pa_c         408

새 변수 코드 점검 (분석 대상, 결측 포함 인원 분포):
incm_c
1.0    1426
2.0    1434
3.0    1434
4.0    1427
NaN      28 

pa_c
0.0    2973
1.0    2368
NaN     408 

완전 사례의 노출별 인원:
breakfast_skip
아침 식사(0)    3693
결식(1)       1579

완전 사례의 가중 비만 유병률(점추정): 37.9%  (흔한 결과 → 오즈비 해석 주의, 유병률비는 12주차)

가중치 비교 (설계 자료 d 기준)
  wt_tot > 0: 6,805명 / wt_itvex > 0: 6,805명 / 둘 다: 6,805명
  분석 대상 안에서 wt_itvex가 결측이거나 0인 사람: 0명
  가중치 합(명): wt_tot 51,522,035 / wt_itvex 50,124,465


## 4. 설계 기반 회귀 함수
11주차 함수 그대로이고, **포아송(로그 연결) 회귀 `kind='poisson'` 분기만 추가**했다(유병률비용). 로지스틱·선형 분기와 분산 추정(층·집락 Taylor 선형화, 자유도 = 집락 수 − 층 수)은 바뀌지 않았다. 5절에서 로지스틱 결과가 11주차와 같은지 확인한다.

In [12]:
from scipy import stats

def fmt_p(p):
    return '<0.001' if p < 0.001 else f'{p:.3f}'

def p_eq(p):
    return 'p <0.001' if p < 0.001 else f'p = {p:.3f}'

def design_matrix(dd, cont, cat):
    """상수항 + 연속형 + 범주형 더미(가장 작은 값이 기준)."""
    X = dd[cont].astype(float).copy()
    for c in cat:
        X = X.join(pd.get_dummies(dd[c].astype(int), prefix=c, drop_first=True, dtype=float))
    X.insert(0, 'const', 1.0)
    return X

def _irls(X, y, w, kind):
    """가중 회귀계수와 (계수, 적합값, 빵 부분 A) 반환. kind: 'logit', 'poisson'(12주차 추가) 또는 'linear'."""
    if kind == 'linear':
        A = (X * w[:, None]).T @ X
        b = np.linalg.solve(A, (X * w[:, None]).T @ y)
        return b, X @ b, A
    if kind == 'poisson':          # 12주차 추가: 로그 연결 포아송(유병률비). 로지스틱과 같은 IRLS 틀
        b = np.zeros(X.shape[1])
        b[0] = np.log(np.average(y, weights=w))      # 상수항(첫 열)에서 시작
        for _ in range(200):
            eta = X @ b
            mu = np.exp(eta)
            A = (X * (w * mu)[:, None]).T @ X
            b_new = np.linalg.solve(A, (X * (w * mu)[:, None]).T @ (eta + (y - mu) / mu))
            done = np.max(np.abs(b_new - b)) < 1e-10
            b = b_new
            if done:
                break
        else:
            raise RuntimeError('포아송 회귀가 수렴하지 않았습니다')
        mu = np.exp(X @ b)
        A = (X * (w * mu)[:, None]).T @ X
        return b, mu, A
    b = np.zeros(X.shape[1])
    for _ in range(100):
        eta = X @ b
        mu = 1 / (1 + np.exp(-eta))
        v = np.clip(mu * (1 - mu), 1e-10, None)
        A = (X * (w * v)[:, None]).T @ X
        b_new = np.linalg.solve(A, (X * (w * v)[:, None]).T @ (eta + (y - mu) / v))
        done = np.max(np.abs(b_new - b)) < 1e-10
        b = b_new
        if done:
            break
    else:
        raise RuntimeError('로지스틱 회귀가 수렴하지 않았습니다')
    mu = 1 / (1 + np.exp(-(X @ b)))
    A = (X * (w * mu * (1 - mu))[:, None]).T @ X
    return b, mu, A

def svy_glm(d, y, cont, cat, dom, kind='logit', wcol='wt_tot'):
    """설계 기반 회귀. d 전체 행(설계 정보 있는 모든 사람)을 두고 dom(True)인 사람만 모형에 넣는다."""
    s = d[dom]
    X = design_matrix(s, cont, cat)
    yy, w = s[y].astype(float).values, s[wcol].values
    b, mu, A = _irls(X.values, yy, w, kind)

    # 점수(score)를 층·집락별로 합산. 도메인 밖 행은 0으로 두되 집락에는 남긴다.
    U = pd.DataFrame(X.values * (w * (yy - mu))[:, None], index=s.index, columns=X.columns)
    U = U.reindex(d.index, fill_value=0.0)
    U['h'], U['c'] = d['kstrata'].values, d['clu'].values
    uc = U.groupby(['h', 'c'])[list(X.columns)].sum().reset_index()
    meat = np.zeros((len(b), len(b)))
    for h, g in uc.groupby('h'):
        n = len(g)
        if n > 1:
            z = g[list(X.columns)].values
            z = z - z.mean(axis=0)
            meat += n / (n - 1) * (z.T @ z)
    Ai = np.linalg.inv(A)
    se = np.sqrt(np.diag(Ai @ meat @ Ai))

    df_ = d['clu'].nunique() - d['kstrata'].nunique()            # 설계 자유도
    t = stats.t.ppf(0.975, df_)
    p = 2 * stats.t.sf(np.abs(b / se), df_)
    out = pd.DataFrame({'b': b, 'se': se, 'lo': b - t * se, 'hi': b + t * se, 'p': p}, index=X.columns)
    if kind == 'logit':
        out['OR'], out['OR_lo'], out['OR_hi'] = np.exp(out['b']), np.exp(out['lo']), np.exp(out['hi'])
    if kind == 'poisson':
        out['PR'], out['PR_lo'], out['PR_hi'] = np.exp(out['b']), np.exp(out['lo']), np.exp(out['hi'])
    return out, int(len(s)), int(df_)

## 5. 주 분석(모형 2) 다시 실행
11주차 Table 2의 모형 2와 **같은 결과**가 나오는지 확인한다. 아래 `MAIN_11`은 11주차 노트북의 출력값이다. 이 셀이 통과해야 아래 민감도분석을 주 분석과 비교할 수 있다.

In [13]:
dom_cc = d['cc'] == 1
res_main, n_main, df_design = svy_glm(d, 'obese', ['breakfast_skip'] + ADJ_CONT, ADJ_CAT, dom_cc, wcol=WT)
e_main = res_main.loc['breakfast_skip']
print(f"주 분석(모형 2): n = {n_main:,}, 결식 OR = {e_main['OR']:.2f} (95% CI {e_main['OR_lo']:.2f}–{e_main['OR_hi']:.2f}), {p_eq(e_main['p'])}  [자유도 {df_design}]")

# 11주차 노트북 출력값 (모형 2)
MAIN_11 = {'n': 5272, 'OR': 1.16, 'lo': 0.99, 'hi': 1.35, 'p': 0.058}
match_11 = (n_main == MAIN_11['n'] and round(e_main['OR'], 2) == MAIN_11['OR']
            and round(e_main['OR_lo'], 2) == MAIN_11['lo'] and round(e_main['OR_hi'], 2) == MAIN_11['hi']
            and round(e_main['p'], 3) == MAIN_11['p'])
print('11주차 결과와 일치:', match_11)
assert match_11, '11주차와 결과가 다릅니다. 앞부분(1~4절) 코드·자료·가중치를 11주차와 비교하세요.'

주 분석(모형 2): n = 5,272, 결식 OR = 1.16 (95% CI 0.99–1.35), p = 0.058  [자유도 165]
11주차 결과와 일치: True


## 6. S-유병률: 포아송(로그 연결) 유병률비 (필수)
비만 유병률이 높아(약 38%) 오즈비가 위험을 실제보다 크게 보일 수 있다. **바꾸는 가정은 효과 측도뿐이다.** 같은 사람(`cc`), 같은 보정변수 8개이고, 연결함수만 logit에서 log로 바꿔 가중 포아송 회귀의 **유병률비(PR)**를 구한다(분산은 같은 설계 기반 샌드위치).

In [14]:
res_pr, n_pr, _ = svy_glm(d, 'obese', ['breakfast_skip'] + ADJ_CONT, ADJ_CAT, dom_cc, kind='poisson', wcol=WT)
e_pr = res_pr.loc['breakfast_skip']
assert n_pr == n_main                                   # 분석 대상을 바꾸지 않으므로 주 분석과 같은 사람

cs = d[dom_cc]
prev_g = {g: (x[WT] * x['obese']).sum() / x[WT].sum() * 100 for g, x in cs.groupby('breakfast_skip')}
print(f"참고) 보정 전 가중 비만 유병률: 아침 식사군 {prev_g[0]:.1f}% / 결식군 {prev_g[1]:.1f}%")
print(f"S-유병률: n = {n_pr:,}, 결식 PR = {e_pr['PR']:.2f} (95% CI {e_pr['PR_lo']:.2f}–{e_pr['PR_hi']:.2f}), {p_eq(e_pr['p'])}")
print(f"같은 보정모형의 OR = {e_main['OR']:.2f} → PR = {e_pr['PR']:.2f} (비만이 흔하면 OR가 PR보다 1에서 더 멀어진다)")

참고) 보정 전 가중 비만 유병률: 아침 식사군 37.0% / 결식군 39.5%
S-유병률: n = 5,272, 결식 PR = 1.09 (95% CI 1.00–1.20), p = 0.057
같은 보정모형의 OR = 1.16 → PR = 1.09 (비만이 흔하면 OR가 PR보다 1에서 더 멀어진다)


## 7. S1: 역방향 인과 (체중 조절 중인 사람 제외)
비만이라서 아침을 거르거나 식습관을 바꾸는 역방향 가능성을 보는 분석이다. **바꾸는 가정은 분석 대상뿐이다.** 모형 2와 같은 모형·보정변수이고, 최근 1년간 체중 **감소**를 노력한 사람을 도메인에서 뺀다(행은 지우지 않고 도메인으로 처리, 층·집락 구조는 그대로 둔다).

- 체중 감소 노력(`BO2_1` = 1)만 뺀다. 유지·증가 노력과 노력해본 적 없음(2~4)은 남긴다.
- 체중 조절 변수가 비해당(8)·모름(9)·결측인 사람은 상태를 알 수 없어 이 분석에서는 뺀다(아래에서 인원을 출력). 팀이 다르게 정하면 `dom_s1` 한 줄만 바꾼다.

In [15]:
WL_LOSE = [1]      # 1 = 체중 감소 노력 (이용지침서 141쪽). 8·9는 비해당·모름이라 아래에서 함께 제외된다
assert d['wl_c'].notna().any(), f'{WL_VAR} 값이 없습니다. 변수명·코드값을 확인하세요.'

print('완전 사례(cc) 안의 체중 조절 여부 분포(집계):')
print(d.loc[dom_cc, 'wl_c'].value_counts(dropna=False).sort_index().to_string())

dom_s1 = dom_cc & d['wl_c'].notna() & ~d['wl_c'].isin(WL_LOSE)
res_s1, n_s1, _ = svy_glm(d, 'obese', ['breakfast_skip'] + ADJ_CONT, ADJ_CAT, dom_s1, wcol=WT)
e_s1 = res_s1.loc['breakfast_skip']
n_out_s1 = n_main - n_s1
print(f"\n주 분석 {n_main:,}명 → S1 {n_s1:,}명 (제외 {n_out_s1:,}명)")
print(f"결식군 인원(비가중): 주 분석 {int((d.loc[dom_cc, 'breakfast_skip'] == 1).sum()):,}명 → S1 {int((d.loc[dom_s1, 'breakfast_skip'] == 1).sum()):,}명")
print(f"S1: n = {n_s1:,}, 결식 OR = {e_s1['OR']:.2f} (95% CI {e_s1['OR_lo']:.2f}–{e_s1['OR_hi']:.2f}), {p_eq(e_s1['p'])}")

완전 사례(cc) 안의 체중 조절 여부 분포(집계):
wl_c
1.0    2148
2.0    1294
3.0     241
4.0    1589

주 분석 5,272명 → S1 3,124명 (제외 2,148명)
결식군 인원(비가중): 주 분석 1,579명 → S1 839명
S1: n = 3,124, 결식 OR = 1.14 (95% CI 0.92–1.41), p = 0.224


## 8. S2: 노출 정의 (아침식사 빈도 네 범주)
결식을 3·4(1) 대 1·2(0)로 나눈 정의가 결과를 좌우하는지 본다. **바꾸는 가정은 노출 정의뿐이다.** 같은 사람(`cc`), 같은 보정변수 8개이고, 노출만 `L_BR_FQ` 네 범주를 그대로 더미로 넣는다(기준: 1). 추세는 1~4를 순서 점수로 넣어 한 단계 늘 때의 오즈비를 본다.

범주 이름은 이용지침서(219쪽 식습관 조사) 기준이다. 4는 '거의 안한다(주 0회)'이다.

In [16]:
BRFQ_LAB = {1: '1 (주 5~7회)', 2: '2 (주 3~4회)', 3: '3 (주 1~2회)', 4: '4 (거의 안 함, 주 0회)'}   # 이용지침서 219쪽

res_s2, n_s2, _ = svy_glm(d, 'obese', ADJ_CONT, ['brfq'] + ADJ_CAT, dom_cc, wcol=WT)
res_s2t, n_s2t, _ = svy_glm(d, 'obese', ['brfq'] + ADJ_CONT, ADJ_CAT, dom_cc, wcol=WT)
assert n_s2 == n_s2t == n_main                           # 분석 대상을 바꾸지 않으므로 주 분석과 같은 사람

print('분석 인원(비가중), 범주별:', {BRFQ_LAB[k]: int(v) for k, v in d.loc[dom_cc, 'brfq'].value_counts().sort_index().items()})
for k in [2, 3, 4]:
    e = res_s2.loc[f'brfq_{k}']
    print(f"  {BRFQ_LAB[k]} 대 {BRFQ_LAB[1]}: OR = {e['OR']:.2f} (95% CI {e['OR_lo']:.2f}–{e['OR_hi']:.2f}), {p_eq(e['p'])}")
et = res_s2t.loc['brfq']
print(f"  추세(한 단계당): OR = {et['OR']:.2f} (95% CI {et['OR_lo']:.2f}–{et['OR_hi']:.2f}), {p_eq(et['p'])}")

분석 인원(비가중), 범주별: {'1 (주 5~7회)': 3173, '2 (주 3~4회)': 520, '3 (주 1~2회)': 631, '4 (거의 안 함, 주 0회)': 948}
  2 (주 3~4회) 대 1 (주 5~7회): OR = 1.55 (95% CI 1.24–1.95), p <0.001
  3 (주 1~2회) 대 1 (주 5~7회): OR = 1.44 (95% CI 1.15–1.79), p = 0.001
  4 (거의 안 함, 주 0회) 대 1 (주 5~7회): OR = 1.23 (95% CI 1.01–1.50), p = 0.035
  추세(한 단계당): OR = 1.08 (95% CI 1.02–1.14), p = 0.014


## 9. S3 (선택): 직접효과 (총에너지 섭취 추가)
에너지 섭취는 DAG에서 **매개 후보**라 주 분석(총효과)에서는 보정하지 않았다. 추가하면 결식이 에너지 섭취를 거치지 않고 비만에 미치는 **직접효과에 가까운 값**을 본다. **바꾸는 가정은 보정집합 한 가지(에너지 섭취 추가)이다.**

에너지 섭취는 영양조사 참여자에게만 값이 있어 **분석 대상이 줄 수 있다.** 그래서 모형 2도 *같은 사람*으로 다시 적합해 짝을 지어 비교한다. 선택 분석이므로 계획표에 넣지 않으면 이 절은 건너뛴다.

In [17]:
dom_s3 = dom_cc & d['en100'].notna()
n_s3 = int(dom_s3.sum())
assert n_s3 > 0, 'N_EN 값이 있는 완전 사례가 없습니다'
print(f'주 분석 {n_main:,}명 → S3 {n_s3:,}명 (N_EN 결측으로 제외 {n_main - n_s3:,}명)')

res_s3ref, _, _ = svy_glm(d, 'obese', ['breakfast_skip'] + ADJ_CONT, ADJ_CAT, dom_s3, wcol=WT)               # 모형 2, 같은 사람
res_s3, _, _ = svy_glm(d, 'obese', ['breakfast_skip', 'en100'] + ADJ_CONT, ADJ_CAT, dom_s3, wcol=WT)        # + 에너지 섭취(100 kcal당)
e_s3ref, e_s3, e_en = res_s3ref.loc['breakfast_skip'], res_s3.loc['breakfast_skip'], res_s3.loc['en100']
print(f"같은 {n_s3:,}명에서 모형 2: 결식 OR = {e_s3ref['OR']:.2f} (95% CI {e_s3ref['OR_lo']:.2f}–{e_s3ref['OR_hi']:.2f}), {p_eq(e_s3ref['p'])}")
print(f"        + 에너지 섭취(S3): 결식 OR = {e_s3['OR']:.2f} (95% CI {e_s3['OR_lo']:.2f}–{e_s3['OR_hi']:.2f}), {p_eq(e_s3['p'])}")
print(f"        에너지 섭취 100 kcal당 OR = {e_en['OR']:.3f} (95% CI {e_en['OR_lo']:.3f}–{e_en['OR_hi']:.3f})")

주 분석 5,272명 → S3 5,271명 (N_EN 결측으로 제외 1명)
같은 5,271명에서 모형 2: 결식 OR = 1.16 (95% CI 0.99–1.35), p = 0.058
        + 에너지 섭취(S3): 결식 OR = 1.20 (95% CI 1.03–1.40), p = 0.021
        에너지 섭취 100 kcal당 OR = 1.020 (95% CI 1.010–1.030)


## 10. D-2 Table 3, 각주, D-3 해석
### 10-1. Table 3
맨 위가 주 분석, 그 아래가 민감도분석이다. 효과 측도는 OR(오즈비) 또는 PR(유병률비)이며, **PR은 OR과 단위가 달라 크기 차이(%)는 계산하지 않고** 결론(방향·유의성)만 비교한다. 분석 대상을 바꾸지 않는 분석(S-유병률, S2)은 분석 인원이 주 분석과 같다.

In [18]:
def verdict(est, lo, hi):
    if lo > 1:
        return '증가·유의'
    if hi < 1:
        return '감소·유의'
    return ('증가' if est > 1 else '감소') + '·유의하지 않음'

def mk(label, change, n, measure, r, var):
    e = r.loc[var]
    est, lo, hi = (e['OR'], e['OR_lo'], e['OR_hi']) if measure == 'OR' else (e['PR'], e['PR_lo'], e['PR_hi'])
    return dict(label=label, change=change, n=n, measure=measure, est=est, lo=lo, hi=hi, p=e['p'])

T3 = [mk('주 분석 (모형 2)', '—', n_main, 'OR', res_main, 'breakfast_skip'),
      mk('S-유병률', '효과 측도: OR → 유병률비(포아송, 로그 연결)', n_pr, 'PR', res_pr, 'breakfast_skip'),
      mk('S1 역방향 인과', '분석 대상: 체중 감소 노력자·체중 조절 여부 미상 제외', n_s1, 'OR', res_s1, 'breakfast_skip')]
for k in [2, 3, 4]:
    T3.append(mk(f'S2 빈도 {BRFQ_LAB[k]} (기준 1)', '노출 정의: 아침식사 빈도 네 범주', n_s2, 'OR', res_s2, f'brfq_{k}'))
T3.append(mk('S2 추세(한 단계당)', '노출 정의: 빈도 순서 점수', n_s2t, 'OR', res_s2t, 'brfq'))
T3.append(mk('S3 직접효과 (선택)', f'보정집합: 에너지 섭취 추가 (분석 대상이 달라 같은 {n_s3:,}명의 모형 2와 비교)', n_s3, 'OR', res_s3, 'breakfast_skip'))
# 선택 분석(S3)을 계획표에 넣지 않으면 위 마지막 줄을 지운다.

m0 = T3[0]
for t in T3:
    t['verdict'] = verdict(t['est'], t['lo'], t['hi'])
    ref_v = verdict(e_s3ref['OR'], e_s3ref['OR_lo'], e_s3ref['OR_hi']) if t['label'].startswith('S3') else m0['verdict']
    t['same'] = '같음' if t['verdict'] == ref_v else '다름'
    if t is m0:
        t['diff'] = '—'
    elif t['label'].startswith('S1'):
        t['diff'] = f"{(t['est'] / m0['est'] - 1) * 100:+.1f}%"
    elif t['label'].startswith('S3'):
        t['diff'] = f"{(t['est'] / e_s3ref['OR'] - 1) * 100:+.1f}% (같은 {n_s3:,}명의 모형 2 OR {e_s3ref['OR']:.2f} 대비)"
    else:
        t['diff'] = '측도·정의가 달라 %는 생략'

table3 = pd.DataFrame([{
    '분석': t['label'], '바꾼 가정': t['change'], '분석 인원 n': f"{t['n']:,}",
    '측도': t['measure'], '추정값': f"{t['est']:.2f}", '95% CI': f"{t['lo']:.2f}–{t['hi']:.2f}", 'p': fmt_p(t['p']),
    '주 분석과의 차이': t['diff'], '결론(주 분석과)': f"{t['verdict']} ({t['same']})"} for t in T3])
print(table3.to_string(index=False))
print()
try:
    print(table3.to_markdown(index=False))
except ImportError:
    print('(to_markdown에 tabulate가 필요합니다: pip install tabulate)')

                           분석                                           바꾼 가정 분석 인원 n 측도  추정값    95% CI      p                          주 분석과의 차이       결론(주 분석과)
                  주 분석 (모형 2)                                               —   5,272 OR 1.16 0.99–1.35  0.058                                  — 증가·유의하지 않음 (같음)
                        S-유병률                    효과 측도: OR → 유병률비(포아송, 로그 연결)   5,272 PR 1.09 1.00–1.20  0.057                    측도·정의가 달라 %는 생략 증가·유의하지 않음 (같음)
                    S1 역방향 인과                 분석 대상: 체중 감소 노력자·체중 조절 여부 미상 제외   3,124 OR 1.14 0.92–1.41  0.224                              -1.9% 증가·유의하지 않음 (같음)
      S2 빈도 2 (주 3~4회) (기준 1)                             노출 정의: 아침식사 빈도 네 범주   5,272 OR 1.55 1.24–1.95 <0.001                    측도·정의가 달라 %는 생략      증가·유의 (다름)
      S2 빈도 3 (주 1~2회) (기준 1)                             노출 정의: 아침식사 빈도 네 범주   5,272 OR 1.44 1.15–1.79  0.001                    측도·정의가 달라 %는 생략      증가·유의 (다름)
S2 빈도 4 (거의 안 함, 주 0회) (기준 1

### 10-2. 각주 (D-2) — 출력 문장을 `docs/12_sensitivity.md` Table 3 각주에 붙인다

In [19]:
adj_names = ', '.join(ADJ_SET[v][0] for v in ADJ_KEYS)
_seen, _ch = set(), []
for t in T3[1:]:
    key = t['change'] if t['label'].startswith('S2') else t['label']
    if key not in _seen:
        _seen.add(key)
        _ch.append(f"{'S2 노출 정의' if t['label'].startswith('S2') else t['label']}: {t['change']}")
changes = '; '.join(_ch)
footnote = (
    f"OR = 오즈비, PR = 유병률비, CI = 신뢰구간. 주 분석은 11주차 모형 2(가중치 {WT}, 층 kstrata, 집락 psu(층+조사구)를 적용한 복합표본 로지스틱 회귀, "
    f"DAG로 정한 보정변수 {len(ADJ_KEYS)}개: {adj_names})이다. 모든 분석에서 표준오차는 Taylor 선형화(샌드위치) 근사로 추정하고 "
    f"95% 신뢰구간과 p값은 자유도 {df_design}(집락 수 − 층 수)의 t분포로 구하였다. 분석 대상은 자료를 삭제하지 않고 도메인으로 지정하여 "
    f"층·집락 구조를 유지하였다. 분석마다 바꾼 점은 하나씩이다 — {changes}. "
    f"S-유병률의 PR은 포아송 회귀(로그 연결, 같은 설계 기반 분산)로, OR와 단위가 달라 크기 차이(%)는 계산하지 않았다. "
    f"결측값은 대체하지 않았고, 분석 대상을 바꾸지 않는 분석은 주 분석과 같은 {n_main:,}명이다. "
    f"자료: 국민건강영양조사 제9기(2022-2024) 중 2024년."
)
print(footnote)

OR = 오즈비, PR = 유병률비, CI = 신뢰구간. 주 분석은 11주차 모형 2(가중치 wt_tot, 층 kstrata, 집락 psu(층+조사구)를 적용한 복합표본 로지스틱 회귀, DAG로 정한 보정변수 8개: 나이, 성별, 교육수준, 소득, 거주지역, 현재흡연, 월간음주, 신체활동(유산소))이다. 모든 분석에서 표준오차는 Taylor 선형화(샌드위치) 근사로 추정하고 95% 신뢰구간과 p값은 자유도 165(집락 수 − 층 수)의 t분포로 구하였다. 분석 대상은 자료를 삭제하지 않고 도메인으로 지정하여 층·집락 구조를 유지하였다. 분석마다 바꾼 점은 하나씩이다 — S-유병률: 효과 측도: OR → 유병률비(포아송, 로그 연결); S1 역방향 인과: 분석 대상: 체중 감소 노력자·체중 조절 여부 미상 제외; S2 노출 정의: 노출 정의: 아침식사 빈도 네 범주; S2 노출 정의: 노출 정의: 빈도 순서 점수; S3 직접효과 (선택): 보정집합: 에너지 섭취 추가 (분석 대상이 달라 같은 5,271명의 모형 2와 비교). S-유병률의 PR은 포아송 회귀(로그 연결, 같은 설계 기반 분산)로, OR와 단위가 달라 크기 차이(%)는 계산하지 않았다. 결측값은 대체하지 않았고, 분석 대상을 바꾸지 않는 분석은 주 분석과 같은 5,272명이다. 자료: 국민건강영양조사 제9기(2022-2024) 중 2024년.


### 10-3. D-3 해석 초안
분석마다 결론(방향·크기·신뢰구간)이 주 분석과 같은지와, 어떤 가정이 결론을 좌우하는지를 숫자로 정리한다. 아래 출력은 **초안**이므로 팀이 검토해 `docs/12_sensitivity.md`와 13주차 보고서용 핵심 문장 두세 개로 다듬는다.

In [20]:
print('[분석별 결론 비교]')
for t in T3:
    print(f"- {t['label']}: {t['measure']} {t['est']:.2f} (95% CI {t['lo']:.2f}–{t['hi']:.2f}, {p_eq(t['p'])}) → {t['verdict']}, 주 분석과 결론 {t['same']}")

diff_rows = [t for t in T3[1:] if t['same'] == '다름']
edge = (0.95 <= m0['lo'] <= 1.05) or (0.95 <= m0['hi'] <= 1.05)
print('\n[어떤 가정이 결론을 좌우하는가]')
if diff_rows:
    print('- 결론이 달라진 분석: ' + ', '.join(f"{t['label']}({t['verdict']})" for t in diff_rows) + '. 해당 가정이 결론에 영향을 준다.')
else:
    print('- 모든 민감도분석에서 주 분석과 결론(방향·유의성)이 같았다.')
if edge:
    print(f"- 주 분석의 신뢰구간({m0['lo']:.2f}–{m0['hi']:.2f})이 1 근처에 걸려 있어 작은 변화에도 유의성 판단이 바뀔 수 있으므로, 유의 여부보다 추정값의 방향과 크기를 중심으로 해석한다.")
s1, s3 = T3[2], T3[-1]
print(f"- 역방향 인과(S1): 체중 감소 노력자 등 {n_out_s1:,}명을 뺀 뒤 OR {s1['est']:.2f} ({s1['diff']}).")
print(f"- 직접효과(S3, 같은 {n_s3:,}명): 모형 2 OR {e_s3ref['OR']:.2f} → 에너지 섭취 추가 후 {e_s3['OR']:.2f}.")

print('\n[핵심 문장 초안 — 팀이 수정]')
print(f"1. 보정 후 아침 결식의 비만 오즈비는 {m0['est']:.2f}(95% CI {m0['lo']:.2f}–{m0['hi']:.2f})였고, 유병률비는 {T3[1]['est']:.2f}(95% CI {T3[1]['lo']:.2f}–{T3[1]['hi']:.2f})로 " + ('오즈비보다 1에 가까웠다.' if abs(np.log(T3[1]['est'])) < abs(np.log(m0['est'])) else '오즈비와 비슷하거나 1에서 더 멀었다.'))
print(f"2. 체중 감소 노력자를 제외한 분석(OR {s1['est']:.2f}, 95% CI {s1['lo']:.2f}–{s1['hi']:.2f})과 아침식사 빈도 네 범주 분석(추세 OR {T3[-2]['est']:.2f}, 95% CI {T3[-2]['lo']:.2f}–{T3[-2]['hi']:.2f})에서 결론은 주 분석과 " + ('같았다.' if not diff_rows else '일부 달랐다.'))
print('3. (한계) 한 시점 자료라 역방향 인과를 완전히 배제할 수 없고, 체중 조절 변수는 최근 1년 노력 여부만 반영한다.')

[분석별 결론 비교]
- 주 분석 (모형 2): OR 1.16 (95% CI 0.99–1.35, p = 0.058) → 증가·유의하지 않음, 주 분석과 결론 같음
- S-유병률: PR 1.09 (95% CI 1.00–1.20, p = 0.057) → 증가·유의하지 않음, 주 분석과 결론 같음
- S1 역방향 인과: OR 1.14 (95% CI 0.92–1.41, p = 0.224) → 증가·유의하지 않음, 주 분석과 결론 같음
- S2 빈도 2 (주 3~4회) (기준 1): OR 1.55 (95% CI 1.24–1.95, p <0.001) → 증가·유의, 주 분석과 결론 다름
- S2 빈도 3 (주 1~2회) (기준 1): OR 1.44 (95% CI 1.15–1.79, p = 0.001) → 증가·유의, 주 분석과 결론 다름
- S2 빈도 4 (거의 안 함, 주 0회) (기준 1): OR 1.23 (95% CI 1.01–1.50, p = 0.035) → 증가·유의, 주 분석과 결론 다름
- S2 추세(한 단계당): OR 1.08 (95% CI 1.02–1.14, p = 0.014) → 증가·유의, 주 분석과 결론 다름
- S3 직접효과 (선택): OR 1.20 (95% CI 1.03–1.40, p = 0.021) → 증가·유의, 주 분석과 결론 다름

[어떤 가정이 결론을 좌우하는가]
- 결론이 달라진 분석: S2 빈도 2 (주 3~4회) (기준 1)(증가·유의), S2 빈도 3 (주 1~2회) (기준 1)(증가·유의), S2 빈도 4 (거의 안 함, 주 0회) (기준 1)(증가·유의), S2 추세(한 단계당)(증가·유의), S3 직접효과 (선택)(증가·유의). 해당 가정이 결론에 영향을 준다.
- 주 분석의 신뢰구간(0.99–1.35)이 1 근처에 걸려 있어 작은 변화에도 유의성 판단이 바뀔 수 있으므로, 유의 여부보다 추정값의 방향과 크기를 중심으로 해석한다.
- 역방향 인과(S1): 체중 감소 노력자 등 2,148명을 뺀 뒤 OR 1.14 (-1.9%)

### 제출 전 점검

In [21]:
# 1) 주 분석이 11주차와 같다
assert match_11
# 2) 분석 대상을 바꾸지 않는 분석은 주 분석과 같은 사람이다
assert n_pr == n_s2 == n_s2t == n_main == int(d['cc'].sum())
# 3) 도메인 분석은 완전 사례 안에 있다 (도메인 밖 사람은 들어오지 않음)
assert (d.loc[dom_s1, 'cc'] == 1).all() and (d.loc[dom_s3, 'cc'] == 1).all()
# 4) S1·S3는 인원이 줄거나 같다
assert n_s1 <= n_main and n_s3 <= n_main
# 5) 설계 변수 유효, 보정변수 열 존재
assert d['wt_tot'].gt(0).all() and d['kstrata'].notna().all() and d['psu'].notna().all() and all(c in d.columns for c in ADJ)
# 6) 보정집합은 11주차 DAG의 8개와 같다 (민감도분석에서 바꾸지 않음)
assert ADJ_KEYS == ['age', 'sex', 'educ', 'incm', 'area', 'smoke', 'drink', 'pa']
# 7) 개인별 행을 출력하지 않았다 (요약 표와 집계값만 출력)
print('점검 통과')

점검 통과
